# Final Facility Tier Model: K-Means Care-Bundle Clustering

This notebook is the final data/model notebook for the New Jersey Behavioral Health Access Navigator. It uses a small New Jersey processed sample derived from `clustered_dataframe_final.csv`, which stays outside GitHub. The notebook engineers service features, runs K-Means clustering, creates elbow and silhouette plots, assigns plain-language tier labels, and explains what features define each tier.

The tier is an interpretation aid for navigators. It does not show real-time appointment availability, book care, decide clinical appropriateness, or replace professional judgment.

## What K-Means Does Here

K-Means is an unsupervised clustering method. It groups facilities by similar service patterns without being given a correct answer in advance. In this project, those patterns come from service setting, type of care, facility type, treatment approaches, emergency services, payment/funding signals, recovery support, age groups, and ancillary services.

The numeric cluster labels are not meaningful by themselves. After clustering, we inspect each cluster's feature profile and attach plain-language tier names that a non-coder can explain.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler

sns.set_theme(style='whitegrid', palette='Set2')

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'facility_service_demo.csv'
OUTPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'facility_tiers_demo.csv'

DATA_PATH

## Load The Processed Demo Data

The committed file is intentionally small: 15 New Jersey rows sampled from the local processed file in Downloads, balanced across the five existing tier labels. The full 8,319-row national processed CSV is not committed to GitHub.

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f'Rows: {df.shape[0]:,} | Columns: {df.shape[1]:,}')
df[['facility_name', 'city', 'state', 'cluster_label', 'tier_name']].head()

## Engineer Service Features

The source columns contain comma-separated service descriptions. We split those descriptions into lists, one-hot encode each observed service signal, and add simple count features. The model does not use facility name or city as clustering inputs because the tier should describe service complexity, not identity or location.

In [ ]:
SERVICE_COLUMNS = [
    'type_of_care',
    'service_setting',
    'facility_type',
    'treatment_approaches',
    'emergency_services',
    'payment_funding',
    'recovery_support',
    'age_groups_accepted',
    'ancillary_services',
]

def split_values(value):
    if pd.isna(value) or str(value).strip() == '':
        return []
    return [item.strip() for item in str(value).split(',') if item.strip()]

feature_frames = []
count_features = pd.DataFrame(index=df.index)

for column in SERVICE_COLUMNS:
    lists = df[column].apply(split_values)
    count_features[f'{column}_count'] = lists.apply(len)

    mlb = MultiLabelBinarizer()
    encoded = pd.DataFrame(
        mlb.fit_transform(lists),
        columns=[f'{column}__{label}' for label in mlb.classes_],
        index=df.index,
    )
    feature_frames.append(encoded)

service_features = pd.concat(feature_frames + [count_features], axis=1)
service_features['total_service_signal_count'] = count_features.sum(axis=1)

scaler = StandardScaler()
X = scaler.fit_transform(service_features)

print(f'Engineered features: {service_features.shape[1]}')
service_features.head()

## Check Cluster Counts

The local processed file already included five tier labels. We keep `k=5` for the demo so the notebook can reproduce the same kind of five-tier story. On the final full processed dataset, the team should re-check the elbow and silhouette plots before deciding whether five clusters still makes sense.

In [ ]:
candidate_ks = range(2, min(8, len(df) - 1) + 1)
model_scores = []

for k in candidate_ks:
    model = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = model.fit_predict(X)
    model_scores.append({
        'k': k,
        'inertia': model.inertia_,
        'silhouette': silhouette_score(X, labels),
    })

scores = pd.DataFrame(model_scores)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.lineplot(data=scores, x='k', y='inertia', marker='o', ax=axes[0])
axes[0].set_title('Elbow check')
axes[0].set_xlabel('Number of clusters (k)')
axes[0].set_ylabel('Within-cluster variation')

sns.lineplot(data=scores, x='k', y='silhouette', marker='o', ax=axes[1])
axes[1].set_title('Silhouette check')
axes[1].set_xlabel('Number of clusters (k)')
axes[1].set_ylabel('Silhouette score')

plt.tight_layout()
scores

## Fit Final Demo Model And Assign Tier Names

We run K-Means with five clusters. Because this small demo sample was drawn from a previously tiered processed file, we map each new cluster to the most common existing plain-language tier name in that cluster. For the final full dataset, the team should inspect cluster profiles and adjust labels if the feature patterns change.

In [ ]:
N_CLUSTERS = 5

kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=20)
df['model_cluster'] = kmeans.fit_predict(X)

cluster_to_tier = (
    df.groupby('model_cluster')['tier_name']
    .agg(lambda values: values.value_counts().idxmax())
    .to_dict()
)
df['model_tier_name'] = df['model_cluster'].map(cluster_to_tier)

df[['facility_name', 'city', 'tier_name', 'model_cluster', 'model_tier_name']].sort_values(['model_tier_name', 'facility_name'])

## Explain What Defines Each Tier

The table below shows the strongest feature signals for each plain-language tier in the demo file. These definitions are descriptive. They do not rank facilities from best to worst. A focused outpatient provider may be the right fit for one person, while a comprehensive support hub may be useful for someone with broader care needs.

In [ ]:
profile_input = pd.concat([df[['model_cluster', 'model_tier_name']], service_features], axis=1)
cluster_profile = profile_input.groupby(['model_cluster', 'model_tier_name'])[service_features.columns].mean()
overall_profile = service_features.mean()

rows = []
for (cluster_id, tier), row in cluster_profile.iterrows():
    lift = (row - overall_profile).sort_values(ascending=False)
    defining_features = [
        feature.replace('__', ': ').replace('_', ' ')
        for feature in lift.head(8).index
    ]
    rows.append({
        'facility_tier': tier,
        'model_cluster': cluster_id,
        'facilities_in_demo': int((df['model_cluster'] == cluster_id).sum()),
        'average_service_signal_count': round(row['total_service_signal_count'], 1),
        'top_defining_features': '; '.join(defining_features),
    })

tier_explanations = pd.DataFrame(rows).sort_values('facility_tier')
tier_explanations

In [ ]:
plt.figure(figsize=(11, 5))
plot_df = df.assign(total_service_signal_count=service_features['total_service_signal_count'])
sns.barplot(data=plot_df, x='model_tier_name', y='total_service_signal_count', estimator='mean', errorbar=None)
plt.title('Average service signal count by assigned facility tier')
plt.xlabel('Facility tier')
plt.ylabel('Average service signal count')
plt.xticks(rotation=25, ha='right')
plt.tight_layout()

## Save Tiered Demo Output

The output keeps the original processed fields and adds `model_cluster` plus `model_tier_name`. The prototype and report should show the plain-language tier, not just the numeric cluster.

In [ ]:
tiered_facilities = df.sort_values(['model_tier_name', 'city', 'facility_name'])
tiered_facilities.to_csv(OUTPUT_PATH, index=False)
print(f'Wrote tiered demo output to: {OUTPUT_PATH}')
tiered_facilities[['facility_name', 'city', 'state', 'model_tier_name', 'model_cluster']].head(10)

## Responsible Use

These tiers summarize service patterns in public data. They cannot confirm appointment availability, new-client acceptance, insurance acceptance for a specific person, or clinical fit. A navigator should use the tier as one clue, then verify details directly with the facility and apply professional judgment.